## 1. Import Required Libraries

- `pandas` is used for reading and manipulating the dataset.
- `numpy` is used for numerical operations.
- `DecisionTreeClassifier` is used to build the classification model.
- `train_test_split` is used to divide the data into training and testing sets.
- `GridSearchCV` is used to test multiple combinations of model parameters.


In [1]:
import pandas as pd
import numpy as np

from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

# Display all columns when viewing a DataFrame
pd.set_option("display.max_columns", None)


## 2. Load the Dataset

The following cell reads `Fiberbits.csv`.

Make sure that the CSV file is stored in the same folder as this notebook.


In [2]:
# Load the dataset
Fiber_df = pd.read_csv("Fiberbits.csv", header=0)

# Display the first three rows
Fiber_df.head(3)


,active_cust,income,months_on_network,Num_complaints,number_plan_changes,relocated,monthly_bill,technical_issues_per_month,Speed_test_result
0,0,1586,85,4,1,0,121,4,85
1,0,1581,85,4,1,0,133,4,85
2,0,1594,82,4,1,0,118,4,85


## 3. Inspect the Dataset

The `info()` method shows:

- Number of rows and columns
- Column names
- Data types
- Number of non-null values
- Approximate memory usage


In [3]:
# Display dataset structure and data types
Fiber_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column                      Non-Null Count   Dtype
---  ------                      --------------   -----
 0   active_cust                 100000 non-null  int64
 1   income                      100000 non-null  int64
 2   months_on_network           100000 non-null  int64
 3   Num_complaints              100000 non-null  int64
 4   number_plan_changes         100000 non-null  int64
 5   relocated                   100000 non-null  int64
 6   monthly_bill                100000 non-null  int64
 7   technical_issues_per_month  100000 non-null  int64
 8   Speed_test_result           100000 non-null  int64
dtypes: int64(9)
memory usage: 6.9 MB


## 4. Check the Dataset Shape and Missing Values

Before building the model, inspect the number of records, number of columns, and missing values.


In [4]:
print("Dataset shape:", Fiber_df.shape)

print("\nMissing values in each column:")
display(Fiber_df.isnull().sum())


Dataset shape: (100000, 9)

Missing values in each column:


active_cust                   0
income                        0
months_on_network             0
Num_complaints                0
number_plan_changes           0
relocated                     0
monthly_bill                  0
technical_issues_per_month    0
Speed_test_result             0
dtype: int64

## 5. Review the Target Variable

The target column is `active_cust`.

For classification, it is useful to inspect the number of records in each target category.


In [5]:
# Display the distribution of the target variable
Fiber_df["active_cust"].value_counts()


active_cust
1    57859
0    42141
Name: count, dtype: int64

## 6. Define Features and Target

- **Features (`X`)**: All columns except `active_cust`
- **Target (`y`)**: The `active_cust` column



because the older positional `axis` syntax is deprecated in recent pandas versions.


In [6]:
# Create a list of feature column names
features = list(Fiber_df.drop(columns=["active_cust"]).columns)

print("Feature columns:")
print(features)

# Create feature matrix X and target vector y
X = Fiber_df[features].to_numpy()
y = Fiber_df["active_cust"].to_numpy()

print("\nX shape:", X.shape)
print("y shape:", y.shape)


Feature columns:
['income', 'months_on_network', 'Num_complaints', 'number_plan_changes', 'relocated', 'monthly_bill', 'technical_issues_per_month', 'Speed_test_result']

X shape: (100000, 8)
y shape: (100000,)


## 7. Split the Data into Training and Testing Sets

The dataset is divided as follows:

- 80% training data
- 20% testing data

`random_state=42` makes the split reproducible.

`stratify=y` maintains approximately the same target-class distribution in both the training and testing sets.


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    train_size=0.80,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)


X_train shape: (80000, 8)
X_test shape : (20000, 8)
y_train shape: (80000,)
y_test shape : (20000,)


# Model 1: Decision Tree with Default Parameters

A Decision Tree model is first created using its default settings.

A default Decision Tree may continue splitting until the leaves are pure or cannot be split further. This can result in:

- Very high training accuracy
- Lower testing accuracy
- Overfitting


In [8]:
# Create the first Decision Tree model
clf1 = DecisionTreeClassifier(random_state=42)

# Train the model
clf1.fit(X_train, y_train)


,criterion,'gini'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


## 8. Evaluate Model 1

The model is evaluated on both training and testing data.

A large difference between training and testing accuracy can indicate overfitting.


In [9]:
# Predict the target values
y_train_pred1 = clf1.predict(X_train)
y_test_pred1 = clf1.predict(X_test)

# Calculate accuracy
train_accuracy1 = accuracy_score(y_train, y_train_pred1)
test_accuracy1 = accuracy_score(y_test, y_test_pred1)

print(f"Model 1 Training Accuracy: {train_accuracy1:.4f}")
print(f"Model 1 Testing Accuracy : {test_accuracy1:.4f}")
print(f"Accuracy Difference      : {train_accuracy1 - test_accuracy1:.4f}")


Model 1 Training Accuracy: 0.9974
Model 1 Testing Accuracy : 0.8470
Accuracy Difference      : 0.1504


## 9. Classification Report for Model 1

The classification report includes:

- **Precision**: Out of all predicted positive records, how many were correct.
- **Recall**: Out of all actual positive records, how many were identified.
- **F1-score**: Harmonic mean of precision and recall.
- **Support**: Number of actual records in each class.


In [10]:
print("Classification Report - Model 1")
print(classification_report(y_test, y_test_pred1))


Classification Report - Model 1
              precision    recall  f1-score   support

           0       0.82      0.82      0.82      8428
           1       0.87      0.87      0.87     11572

    accuracy                           0.85     20000
   macro avg       0.84      0.84      0.84     20000
weighted avg       0.85      0.85      0.85     20000



## 10. Confusion Matrix for Model 1

The confusion matrix shows the number of correct and incorrect predictions for every target class.


In [11]:
confusion_matrix_model1 = confusion_matrix(y_test, y_test_pred1)

print("Confusion Matrix - Model 1")
print(confusion_matrix_model1)


Confusion Matrix - Model 1
[[ 6930  1498]
 [ 1562 10010]]


# Model 2: Decision Tree with Manually Tuned Parameters

The second model limits the complexity of the Decision Tree.

### Parameters

- `criterion='gini'`: Measures the quality of a split using Gini impurity.
- `splitter='best'`: Selects the best available split.
- `max_depth=20`: Limits the maximum depth of the tree.
- `min_samples_split=5`: Requires at least five samples to split a node.
- `min_samples_leaf=5`: Requires at least five samples in each leaf node.
- `max_leaf_nodes=10`: Limits the total number of leaf nodes.

These restrictions can reduce overfitting.


In [12]:
clf2 = DecisionTreeClassifier(
    criterion="gini",
    splitter="best",
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=5,
    max_leaf_nodes=10,
    random_state=42
)

# Train Model 2
clf2.fit(X_train, y_train)


,criterion,'gini'
,splitter,'best'
,max_depth,20
,min_samples_split,5
,min_samples_leaf,5
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,10
,min_impurity_decrease,0.0
,class_weight,None


## 11. Evaluate Model 2

The training and testing accuracies are calculated and compared with Model 1.


In [13]:
# Predictions from Model 2
y_train_pred2 = clf2.predict(X_train)
y_test_pred2 = clf2.predict(X_test)

# Accuracy scores
train_accuracy2 = accuracy_score(y_train, y_train_pred2)
test_accuracy2 = accuracy_score(y_test, y_test_pred2)

print(f"Model 2 Training Accuracy: {train_accuracy2:.4f}")
print(f"Model 2 Testing Accuracy : {test_accuracy2:.4f}")
print(f"Accuracy Difference      : {train_accuracy2 - test_accuracy2:.4f}")


Model 2 Training Accuracy: 0.8343
Model 2 Testing Accuracy : 0.8351
Accuracy Difference      : -0.0009


## 12. Compare Model 1 and Model 2

A model with slightly lower training accuracy but better or similar testing accuracy may generalize better to unseen data.


In [14]:
model_comparison = pd.DataFrame({
    "Model": ["Default Decision Tree", "Manually Tuned Decision Tree"],
    "Training Accuracy": [train_accuracy1, train_accuracy2],
    "Testing Accuracy": [test_accuracy1, test_accuracy2],
    "Train-Test Difference": [
        train_accuracy1 - test_accuracy1,
        train_accuracy2 - test_accuracy2
    ]
})

model_comparison


,Model,Training Accuracy,Testing Accuracy,Train-Test Difference
0,Default Decision Tree,0.997413,0.84700,0.150413
1,Manually Tuned Decision Tree,0.834275,0.83515,-0.000875


# Grid Search for Optimal Parameters

`GridSearchCV` systematically tests different combinations of parameters.

The following values are tested:

- `criterion`: `gini`, `entropy`
- `max_depth`: 2 through 9

The model is evaluated using 10-fold cross-validation and ROC-AUC scoring.


In [15]:
# Define the parameter combinations
tuned_parameters = {
    "criterion": ["gini", "entropy"],
    "max_depth": range(2, 10)
}

# Base Decision Tree model
base_tree = DecisionTreeClassifier(random_state=42)

# Configure GridSearchCV
grid_search = GridSearchCV(
    estimator=base_tree,
    param_grid=tuned_parameters,
    cv=10,
    scoring="roc_auc",
    return_train_score=True,
    n_jobs=-1
)

# Run the grid search
grid_search.fit(X_train, y_train)


,estimator,DecisionTreeC...ndom_state=42)
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_depth': range(2, 10)}"
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,criterion,'entropy'


## 13. Display the Best Cross-Validation Score

`best_score_` returns the highest average ROC-AUC score found during cross-validation.


In [16]:
print(f"Best Cross-Validation ROC-AUC Score: {grid_search.best_score_:.4f}")


Best Cross-Validation ROC-AUC Score: 0.9343


## 14. Display the Best Parameters

`best_params_` returns the parameter combination that produced the highest cross-validation score.


In [17]:
print("Best Parameters:")
print(grid_search.best_params_)


Best Parameters:
{'criterion': 'entropy', 'max_depth': 9}


## 15. View All Grid Search Results

The old property `grid_scores_` is no longer available in current versions of scikit-learn.

It has been replaced with `cv_results_`.

The following table displays the tested parameters, mean training score, mean validation score, score variation, and rank.


In [18]:
grid_results = pd.DataFrame(grid_search.cv_results_)

selected_results = grid_results[
    [
        "param_criterion",
        "param_max_depth",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score")

selected_results


,param_criterion,param_max_depth,mean_train_score,mean_test_score,std_test_score,rank_test_score
15,entropy,9,0.941493,0.934296,0.002396,1
7,gini,9,0.941410,0.934209,0.002455,2
6,gini,8,0.934287,0.929727,0.002478,3
14,entropy,8,0.932569,0.928229,0.002921,4
5,gini,7,0.922683,0.919927,0.002544,5
13,entropy,7,0.920529,0.918090,0.003012,6
4,gini,6,0.910338,0.908229,0.003409,7
12,entropy,6,0.908498,0.906806,0.003323,8
11,entropy,5,0.893053,0.891967,0.003269,9
3,gini,5,0.891964,0.890777,0.003355,10


# Final Grid-Search Model Evaluation

`best_estimator_` returns the Decision Tree model trained with the best parameter combination found by Grid Search.


In [19]:
best_tree = grid_search.best_estimator_

# Make predictions
y_train_pred_best = best_tree.predict(X_train)
y_test_pred_best = best_tree.predict(X_test)

# Calculate accuracy
best_train_accuracy = accuracy_score(y_train, y_train_pred_best)
best_test_accuracy = accuracy_score(y_test, y_test_pred_best)

print(f"Best Tree Training Accuracy: {best_train_accuracy:.4f}")
print(f"Best Tree Testing Accuracy : {best_test_accuracy:.4f}")


Best Tree Training Accuracy: 0.8736
Best Tree Testing Accuracy : 0.8710


## 16. Calculate ROC-AUC on the Test Data

For binary classification, ROC-AUC is calculated using the predicted probability of the positive class.

A score closer to 1 indicates better class-separation ability.


In [20]:
# Check whether the problem is binary classification
if len(np.unique(y)) == 2:
    y_test_probability = best_tree.predict_proba(X_test)[:, 1]
    test_roc_auc = roc_auc_score(y_test, y_test_probability)
    print(f"Best Tree Test ROC-AUC: {test_roc_auc:.4f}")
else:
    print("Binary ROC-AUC calculation skipped because the target has more than two classes.")


Best Tree Test ROC-AUC: 0.9363


## 17. Final Model Comparison

The following table compares all three Decision Tree models.


In [21]:
final_comparison = pd.DataFrame({
    "Model": [
        "Default Decision Tree",
        "Manually Tuned Decision Tree",
        "Grid Search Decision Tree"
    ],
    "Training Accuracy": [
        train_accuracy1,
        train_accuracy2,
        best_train_accuracy
    ],
    "Testing Accuracy": [
        test_accuracy1,
        test_accuracy2,
        best_test_accuracy
    ],
    "Train-Test Difference": [
        train_accuracy1 - test_accuracy1,
        train_accuracy2 - test_accuracy2,
        best_train_accuracy - best_test_accuracy
    ]
})

final_comparison


,Model,Training Accuracy,Testing Accuracy,Train-Test Difference
0,Default Decision Tree,0.997413,0.84700,0.150413
1,Manually Tuned Decision Tree,0.834275,0.83515,-0.000875
2,Grid Search Decision Tree,0.873637,0.87100,0.002637


# Conclusion

- The default Decision Tree may produce very high training accuracy and lower testing accuracy, which suggests overfitting.
- Restricting tree complexity can improve generalization.
- Grid Search provides a systematic method for selecting suitable model parameters.
- Testing accuracy should not be evaluated alone. Precision, recall, F1-score, confusion matrix, and ROC-AUC should also be considered.
- The exact scores depend on the contents of `Fiberbits.csv` and the train-test split.
